| Input / Situation                     | What Could Go Wrong?                                    | Desired Behavior                              |
| ------------------------------------- | ------------------------------------------------------- | --------------------------------------------- |
| Empty data (`""`)                     | Generates a meaningless QR with no useful payload       | Reject with a clear `ValueError`              |
| Data containing only spaces (`"   "`) | Looks non-empty but carries no meaningful information   | Reject with a clear error                     |
| Extremely large data                  | QR payload becomes too large to encode or scan reliably | Reject with a size-limit error                |
| Unicode text (`"Pay José €10 😊"`)    | Encoding/decoding corruption                            | Should encode and decode correctly            |
| `n_bits = 0`                          | Empty tag, invalid secret string                        | Reject with `ValueError`                      |
| `n_bits < 0`                          | Invalid tag length                                      | Reject with `ValueError`                      |
| `n_bits = 1`                          | Works but weak tamper detection                         | Allow, but document limitations               |
| Very large `n_bits` (64, 128, 256)    | Creates impractical quantum circuits                    | Reject or cap at a reasonable limit           |
| `key=None`                            | No key supplied                                         | Use `get_key()` fallback                      |
| Key passed as `str`                   | HMAC expects bytes and may throw confusing errors       | Convert to bytes or raise clear error         |
| Key passed as integer/list            | Type error deep inside HMAC                             | Raise clear error immediately                 |
| Nonce generation failure              | QR cannot be generated                                  | Propagate a clear exception                   |
| Output path folder does not exist     | File write fails                                        | Create parent folders automatically           |
| Output file already exists            | Existing QR overwritten accidentally                    | Allow overwrite or document behavior          |
| Corrupted QR image later              | Verification fails unexpectedly                         | Return clear verification error               |
| Invalid Base64 payload                | Decoder crashes                                         | Raise informative exception                   |
| Missing payload fields                | Verification logic breaks                               | Validate schema before use                    |
| Modified version field                | Future compatibility issues                             | Check version and reject unsupported versions |
| Modified nonce                        | Tag verification fails                                  | Detect as tampering                           |
| Modified data                         | Tag verification fails                                  | Detect as tampering                           |
| Modified tag                          | Tag verification fails                                  | Detect as tampering                           |
| Replay of an old valid QR             | Integrity still passes                                  | Out of scope for current design               |
| Missing environment variable key      | Verification impossible without fallback                | Use documented demo key                       |
| Different keys on issuer/verifier     | Every QR appears tampered                               | Verification should fail clearly              |


In [1]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))

In [2]:
from quantum_qr.generator import generate

result = generate(
    "pay alice $10",
    "../data/test_small.png"
)

print("Success")

Counts: {'01000110110010001100011010010011010001110011100000111000001100011001110111101110101000101011110001111101110000001110010111011101': 1}

128 Random Bits (List):
[1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0]

128 Random Bits (Hexadecimal):
bba703be3d4577b98c1c1ce2c9631362
Success


In [4]:
from quantum_qr.generator import generate
from quantum_qr.qr_io import read_qr_code
from quantum_qr.payload import decode_payload

emoji_text = "pay 🤝 ₹500"

result = generate(
    emoji_text,
    "../data/unicode_emoji.png"
)

qr_string = read_qr_code(
    "../data/unicode_emoji.png"
)

decoded_payload = decode_payload(
    qr_string
)

print("Original:")
print(emoji_text)

print("\nDecoded:")
print(decoded_payload["data"])

assert decoded_payload["data"] == emoji_text

print("\n✓ Emoji round-trip successful")

Counts: {'01111011101000000100111110100000100010001010010011001010111110110001001010101101010001000010101110001011101000001001001001111010': 1}

128 Random Bits (List):
[0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0]

128 Random Bits (Hexadecimal):
5e4905d1d422b548df53251105f205de
Original:
pay 🤝 ₹500

Decoded:
pay 🤝 ₹500

✓ Emoji round-trip successful


In [6]:
unicode_text = "café münchen 日本語"

result = generate(
    unicode_text,
    "data/unicode_multilingual.png"
)

qr_string = read_qr_code(
    "data/unicode_multilingual.png"
)

decoded_payload = decode_payload(
    qr_string
)

print("Original:")
print(unicode_text)

print("\nDecoded:")
print(decoded_payload["data"])

assert decoded_payload["data"] == unicode_text

print("\n✓ Unicode round-trip successful")

Counts: {'11010110010110101010110010011010100000111000101110010100000100000011001001101000011011011000101000111000100111010011100110001010': 1}

128 Random Bits (List):
[0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1]

128 Random Bits (Hexadecimal):
519cb91c51b6164c0829d1c159355a6b
Original:
café münchen 日本語

Decoded:
café münchen 日本語

✓ Unicode round-trip successful


In [7]:
assert result["payload"]["data"] == unicode_text
assert decoded_payload["data"] == unicode_text

In [2]:
from quantum_qr.generator import generate

fixed_nonce = "0123456789abcdef0123456789abcdef"

result1 = generate(
    data="pay alice $10",
    output_path="../data/fixed1.png",
    key=b"TEST_KEY",
    nonce=fixed_nonce
)

result2 = generate(
    data="pay alice $10",
    output_path="../data/fixed2.png",
    key=b"TEST_KEY",
    nonce=fixed_nonce
)

print(result1["payload"])
print(result2["payload"])

{'version': '1', 'data': 'pay alice $10', 'nonce': '0123456789abcdef0123456789abcdef', 'tag': '01010111'}
{'version': '1', 'data': 'pay alice $10', 'nonce': '0123456789abcdef0123456789abcdef', 'tag': '01010111'}


In [3]:
assert result1["payload"]["nonce"] == result2["payload"]["nonce"]

assert result1["payload"]["tag"] == result2["payload"]["tag"]

print("✓ Deterministic generation works")

✓ Deterministic generation works
